# Open3D 单帧点云可视化（VNC 远程桌面版）

## 关键：先设置环境变量再启动 kernel

参考 `/home/kin/workspace/notes/robot/sceneflow/opensceneflow/vnc-remote-visualization-guide.md`，VNC 里跑 Open3D 必须满足：

1. `DISPLAY=:1` —— 对应 vncserver `:1`
2. `LD_LIBRARY_PATH=/opt/conda/envs/opensf/lib:$LD_LIBRARY_PATH` —— 避免 `GLIBCXX_3.4.29 not found`
3. 禁用 Jupyter 自动启用的 WebRTC backend，强制原生 X11 窗口

**最稳妥的启动方式**（在 VNC 的终端里执行）：

```bash
conda activate opensf
export DISPLAY=:1
export LD_LIBRARY_PATH=/opt/conda/envs/opensf/lib:$LD_LIBRARY_PATH
export OPEN3D_ENABLE_WEBRTC=0
jupyter lab /home/kin/workspace/OpenSceneFlow/visual.ipynb
```

如果 Jupyter 已经启动，下面的 cell 会尝试在 kernel 里设置这些变量，但 **LD_LIBRARY_PATH 在进程启动后可能不生效**。此时请重启 kernel，或改用上面的命令重新启动 Jupyter。

In [1]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import time

from src.dataset import HDF5Dataset
from src.utils.o3d_view import O3DVisualizer

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def removeGround(pc, gm):
    assert len(pc) == len(gm)
    return pc[~gm][:, :3]

def removeFarPoint(pc, threshold=50):
    mask = np.linalg.norm(pc[:,:3], axis=1) > threshold
    return pc[~mask][:, :3]

In [3]:
# 读取 HDF5 数据集；n_frames 多少不影响单帧可视化，默认 2 即可
DATA_DIR = "/home/kin/data/av2/h5py/sensor/train"
dataset = HDF5Dataset(DATA_DIR, n_frames=2)
print(f"数据集长度: {len(dataset)}")

----[Debug] Loading data with num_frames=2, ssl_label=None, eval=False, leaderboard_version=1
数据集长度: 110071


In [4]:
# 读取第 0 帧
data = dataset[0]
print(f"scene_id: {data['scene_id']}, timestamp: {data['timestamp']}")
print(f"可用字段: {list(data.keys())}")

# 取出 pc0：形状一般是 (N, 4) —— x, y, z, intensity（如果只取前3维用于显示）
pc0 = data["pc0"]
print(f"pc0 shape: {pc0.shape}")

scene_id: 58d01358-5927-36fa-9e11-d18d1dc1f4f0, timestamp: 315970823159798000
可用字段: ['scene_id', 'timestamp', 'eval_flag', 'pc0', 'gm0', 'pose0', 'pose1', 'pc1', 'gm1', 'ego_motion', 'lidar_dt', 'flow', 'flow_is_valid', 'flow_category_indices', 'flow_instance_id', 'dufo']
pc0 shape: (97944, 3)


In [ ]:
# =========================================================
# 可视化（使用子进程避免 Open3D GLFW 生命周期问题）
# =========================================================
# Open3D 的 Visualizer 在 destroy_window 时会调用 glfwTerminate()，
# 导致同一个 kernel 内第二次创建 Visualizer 时窗口空白。
# 解决方案：每次可视化在独立子进程里运行。

import multiprocessing as mp


def _show_pcd(points: np.ndarray, point_size: float):
    """在独立进程里显示点云。"""
    import os
    os.environ.setdefault("DISPLAY", ":1")

    import open3d as o3d

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points[:, :3])

    # 按高度着色
    z = points[:, 2]
    z_min, z_max = z.min(), z.max()
    z_norm = (z - z_min) / (z_max - z_min) if z_max > z_min else np.zeros_like(z)
    import matplotlib.pyplot as plt
    colors = plt.get_cmap("viridis")(z_norm)[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors)

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=2.0)

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="pc0", width=1280, height=720)
    vis.add_geometry(pcd)
    vis.add_geometry(frame)

    opt = vis.get_render_option()
    opt.point_size = point_size

    vis.reset_view_point(True)
    vis.update_renderer()

    while vis.poll_events():
        vis.update_renderer()
        time.sleep(0.01)

    vis.destroy_window()


# 过滤点云
pc0_filtered = removeGround(pc0, data["gm0"])
pc0_filtered = removeFarPoint(pc0_filtered)

point_size = 1.0  # 0.5 很细，1.0 适中，3.0 较粗

print(f"正在子进程里打开窗口（point_size={point_size}，点数={len(pc0_filtered)}）...")
process = mp.Process(target=_show_pcd, args=(pc0_filtered, point_size))
process.start()
process.join()
print("子进程已结束，窗口关闭")